In [1]:
# Identify work-related and non-work related ER visits in SyS data using NLP
# Input data are discharge Dx codes, CC, Triage notes, clinical impressions, admit reasons, separated by "--------------------------".

import os 
import re

import pyarrow as pa
import pandas as pd
import numpy as np


import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.cm as cm

import sklearn
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn import metrics
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import SGDClassifier
from sklearn.linear_model import LogisticRegression

import nltk 
from nltk.tokenize import word_tokenize
from nltk.tokenize import RegexpTokenizer
from nltk.corpus import stopwords
from nltk.stem import wordnet
from nltk.stem import WordNetLemmatizer 
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\causti1\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\causti1\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\causti1\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\causti1\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [2]:
################################################
# Functions
################################################

# “PresenceAbsenceVector” model: Converts text to vector using Presence and Absence of Words, and then classify

def PresenceAbsenceVector(token_lst):
    word_set = set()
    vec = []
    for cell in token_lst:
        word_set = word_set.union(set(cell))
    
    for cell in token_lst:
        freq = dict.fromkeys(word_set, 0)
        for word in cell:
            freq[word] = 1
        vec.append(freq)
        
    #print(word_set)
    res = []
    for i in vec:    # convert dict values list to list
        x = []
        for j in i:
            x.append(i[j])
        res.append(x)
        
    return res



# “CountVector” model: Converts text to vector using Count of Words, and then classify
def CountVector(token_lst):
    word_set = set()
    vec = []
    for cell in token_lst:
        word_set = word_set.union(set(cell))
    
    for cell in token_lst:
        freq = dict.fromkeys(word_set, 0)
        for word in cell:
            freq[word] += 1
        vec.append(freq)
    
    #print(word_set)
    res = []
    for i in vec:    # convert dict values list to list
        x = []
        for j in i:
            x.append(i[j])
        res.append(x)
        
    return res



# “TF-IDFVector” model: Converts text to vector using TF-IDF vectorization, and then classify
def TFIDFVector(str_lst):
    tf_vect = TfidfVectorizer(min_df=1, lowercase=True, stop_words="english")
    tf_matrix = tf_vect.fit_transform(str_lst)
    tf_names = tf_vect.get_feature_names_out()
    tf_df = pd.DataFrame(tf_matrix.toarray(), columns=tf_names)
    #print("\n", tf_df)
    
    res = []
    for row in tf_df.index:
        dic = dict.fromkeys(tf_names, 0)
        for col in tf_df: 
            dic[col] = tf_df[col][row]
        res.append(dic)
        
    return res, tf_df



# Provide results based on the prediction
def ConfusionMatrix(test, pred):
    confusion_matrix = metrics.confusion_matrix(test, pred)
    print("Confusion Matrix: ", "\n", confusion_matrix, "\n")
    
    accuracy = metrics.accuracy_score(test, pred)  
    TP = confusion_matrix[0][0]
    FN = confusion_matrix[0][1]
    FP = confusion_matrix[1][0]
    TN = confusion_matrix[1][1]
    TPR = TP / (TP + FN)
    FPR = FP / (FP + TN)
    TNR = TN / (TN + FP)
    PPV = TP / (TP + FP)
    FNR = FN / (FN + TP)
    NER = min(TN + FP, FN + TP) / (TP + TN + FP + FN)
    
    print("Accuracy: (TP + TN) / (P + N) =", accuracy)  
    print("Precision: TP / (TP + FP) =", PPV)
    print("True Positive Rate (Sensitivity): TP / (TP + FN) =", TPR)
    print("True Negative Rate (Specificity): TN / (TN + FP) =", TNR)
    
    print("Misclassification Rate: 1 - Accuracy =", 1-accuracy)
    print("False Positive Rate (false alarm ratio): FP / (FP + TN) =",  FPR)
    print("False Negative Rate (miss rate): FN / (FN + TP) =", FNR)
    print("Null Error Rate: min(TN + FP, FN + TP) / sample size = ", NER)


In [3]:
# Load data and briefly sumarize
os.chdir("\\\\fs1.hhss.local\\edv\\SYS NLP\\Data")

# Load combined data from parquet file
combined_train = pd.read_parquet("\\\\fs1.hhss.local\\edv\\SYS NLP\\Data\\sys_er_combined.parquet")
data_test = pd.read_parquet("\\\\fs1.hhss.local\\edv\\SYS NLP\\Data\\data_test.parquet")
#data_test['Indicator'] = None  # Assign NaN to the 'Indicator' column

# Preprocess for NLP

# Combine training and testing 
len_train = len(combined_train.index) 
len_test = len(data_test.index)

# Note:replace NaN values with an empty string to avoid TypeError in Tokenization
combined = pd.concat([combined_train, data_test[["text", "Indicator"]]], ignore_index=True, sort=False)
txt = combined["text"].fillna("")


# Tokenize
tokenizer = RegexpTokenizer(r'(?u)\W+|\$[\d\.]+|\S+')
tokened = []
for cell in txt:
    tokens = tokenizer.tokenize(cell)
    tokened.append(tokens)

combined["Tokened"] = tokened


# Remove stop words
stop_words = set(stopwords.words("english"))
stop_removed = []
for cell in tokened:
    stop_removed.append([w for w in cell if not w.lower() in stop_words])

combined["Stop Words Removed"] = stop_removed


# Remove punctuations (keep "-")
punc_removed = []
for cell in stop_removed:
    punc_removed.append([w for w in cell if w not in "!\"#$%&'()*+,./:;<=>?@[\\]^_`{|}~" and not w.isspace()])

combined["Punctuations Removed"] = punc_removed


# Assign POS tags
pos = []
for cell in combined["Punctuations Removed"]:
    pos.append(nltk.pos_tag(cell))
    
combined["POS Tags"] = pos


# Lemmatization 
lemmatizer = WordNetLemmatizer()
lemmatized = []
for cell in pos:
    lemm_lst = []
    for tup in cell:
        s = ""
        if tup[1].startswith("N"):
            s = lemmatizer.lemmatize(tup[0], pos="n")
        elif tup[1].startswith("V"):
            s = lemmatizer.lemmatize(tup[0], pos="v")
        elif tup[1].startswith("R"):
            s = lemmatizer.lemmatize(tup[0], pos="r")
        elif tup[1].startswith("J"):
            s = lemmatizer.lemmatize(tup[0], pos="a")
        else:
            s = tup[0]
        lemm_lst.append(s)
    lemmatized.append(lemm_lst)

combined["Lemmatized"] = lemmatized

combined.to_csv("combined.csv")

#combined.head()
combined.tail()

,text,Indicator,Tokened,Stop Words Removed,Punctuations Removed,POS Tags,Lemmatized
54750,K8590 N3000 R748 ------------------- don t fe...,NaN,"[ , K8590, , N3000, , R748, ---------------...","[ , K8590, , N3000, , R748, ---------------...","[K8590, N3000, R748, ------------------- , fe...","[(K8590, NNP), (N3000, NNP), (R748, NNP), ( --...","[K8590, N3000, R748, ------------------- , fe..."
54751,H109 B9689 J069 ------------------- eye issue...,NaN,"[ , H109, , B9689, , J069, ----------------...","[ , H109, , B9689, , J069, ----------------...","[H109, B9689, J069, ------------------- , eye...","[(H109, NNP), (B9689, NNP), (J069, NNP), ( ---...","[H109, B9689, J069, ------------------- , eye..."
54752,R079 E876 ------------------- ems chest pain ...,NaN,"[ , R079, , E876, ------------------- , ems,...","[ , R079, , E876, ------------------- , ems,...","[R079, E876, ------------------- , ems, chest...","[(R079, NNP), (E876, NNP), ( -----------------...","[R079, E876, ------------------- , em, chest,..."
54753,W19XXXA S0990XA Y92009 W08XXXA --------------...,NaN,"[ , W19XXXA, , S0990XA, , Y92009, , W08XXXA...","[ , W19XXXA, , S0990XA, , Y92009, , W08XXXA...","[W19XXXA, S0990XA, Y92009, W08XXXA, ---------...","[(W19XXXA, NNP), (S0990XA, NNP), (Y92009, NNP)...","[W19XXXA, S0990XA, Y92009, W08XXXA, ---------..."
54754,T171XXA W448XXA ------------------- foreign o...,NaN,"[ , T171XXA, , W448XXA, ------------------- ...","[ , T171XXA, , W448XXA, ------------------- ...","[T171XXA, W448XXA, ------------------- , fore...","[(T171XXA, NNP), (W448XXA, NNP), ( -----------...","[T171XXA, W448XXA, ------------------- , fore..."


In [4]:
# Prepare result dataframe

modelResults = data_test.copy(deep=True)



In [6]:
# Naive Bayes Model 
sent_list = [" ".join(x) for x in combined["Lemmatized"]]

# Count Vector
count_vect = CountVectorizer(lowercase=True, stop_words="english", min_df=2)
print(count_vect)
X_count_vect = count_vect.fit_transform(sent_list)
X_names = count_vect.get_feature_names_out()
X_count_vect = pd.DataFrame(X_count_vect.toarray(), columns=X_names)

# Assign training and testing data
X_train_cv = X_count_vect.loc[: len_train - 1]
X_test_cv = X_count_vect.loc[len_train :]
y_train_cv = combined["Indicator"].loc[: len_train - 1]
y_test_cv = combined["Indicator"].loc[len_train :]


nb_cv = MultinomialNB()
nb_cv.fit(X_train_cv, y_train_cv)
y_pred_cv = nb_cv.predict(X_test_cv)

#report_names = ["Work-related", "Non-work related"]
#print("\n-----------------------------------")
#print(  "Count Vector Classification Report:")
#print(  "-----------------------------------")
#print("Accuracy:", accuracy_score(y_pred_cv, y_test_cv), "\n")
#print(classification_report(y_test_cv, y_pred_cv, target_names=report_names))

#ConfusionMatrix(y_test_cv, y_pred_cv)



# TD-IDF Vector
tf_vect = TfidfVectorizer(min_df=1, lowercase=True, stop_words="english")
tf_matrix = tf_vect.fit_transform(sent_list)
tf_names = tf_vect.get_feature_names_out()
tf_df_vect = pd.DataFrame(tf_matrix.toarray(), columns=tf_names)

#X_train_tf, X_test_tf, y_train_tf, y_test_tf = train_test_split(tf_df_vect, combined["Indicator"], test_size=0.2, random_state=5)

# Assign training and testing data
X_train_tf = tf_df_vect.loc[: len_train - 1]
X_test_tf = tf_df_vect.loc[len_train :]
y_train_tf = combined["Indicator"].loc[: len_train - 1]
y_test_tf = combined["Indicator"].loc[len_train :]

nb_tf = MultinomialNB()
nb_tf.fit(X_train_tf, y_train_tf)
y_pred_tf = nb_tf.predict(X_test_tf)


#report_names = ["Work-related", "Non-work related"]
#print("\n------------------------------------")
#print(  "TF_IDF Vector Classification Report:")
#print(  "------------------------------------")
#print("Accuracy:", accuracy_score(y_pred_tf, y_test_tf), "\n")
#print(classification_report(y_test_tf, y_pred_tf, target_names=report_names))

#ConfusionMatrix(y_test_tf, y_pred_tf)

modelResults["Naive Bayes - Count Vector"] = y_pred_cv
modelResults["Naive Bayes - TF-IDF"] = y_pred_tf

#plotting
#fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 6))
#fig.suptitle("Receiver Operating Characteristic", fontsize=16)

#fpr_cv, tpr_cv, _ = metrics.roc_curve(y_test_cv, y_pred_cv)
#roc_auc_cv = metrics.auc(fpr_cv, tpr_cv)

#fpr_tf, tpr_tf, _ = metrics.roc_curve(y_test_tf, y_pred_tf)
#roc_auc_tf = metrics.auc(fpr_tf, tpr_tf)

#ax1.plot(fpr_cv, tpr_cv, "b", label="AUC="+str(roc_auc_cv))
#ax1.plot([0, 1], [0, 1],'r--')
#ax1.set_ylabel('True Positive Rate')
#ax1.set_xlabel('False Positive Rate')
#ax1.legend(loc="best")
#ax1.set_title("Count Vector", fontdict={"fontsize": 12})

#ax2.plot(fpr_tf, tpr_tf, "b", label="AUC="+str(roc_auc_tf))
#ax2.plot([0, 1], [0, 1],'r--')
#ax2.set_ylabel('True Positive Rate')
#ax2.set_xlabel('False Positive Rate')
#ax2.legend(loc="best")
#ax2.set_title("TF-IDF Vector", fontdict={"fontsize": 12})

#plt.show()


CountVectorizer(min_df=2, stop_words='english')


In [5]:
# Linear Support Vector Machine, using a pipline:
# Cuont Vector => TF_IDF Vector => Classifier

X = [" ".join(x) for x in combined["Lemmatized"]]
y = combined["Indicator"]


# Assign training and testing data
X_train = X[: len_train]
X_test = X[len_train :]
y_train = y.loc[: len_train - 1]
y_test = y.loc[len_train :]

sgd = Pipeline([('vect', CountVectorizer()),
                ('tfidf', TfidfTransformer()),
                ('clf', SGDClassifier(loss='hinge', penalty='l2',alpha=1e-3, random_state=42, max_iter=5, tol=None)),
               ])

sgd.fit(X_train, y_train)
y_pred = sgd.predict(X_test)

#report_names = ["Reportable", "Non-reportable"]
#print("Accuracy:", accuracy_score(y_pred, y_test))
#print("\n----------------------------------------------------")
#print(  "Linear Support Vector Machine Classification Report:")
#print(  "----------------------------------------------------")
#print(classification_report(y_test, y_pred, target_names=report_names))

#ConfusionMatrix(y_test, y_pred)

modelResults["Linear Support Vector Machine - TF-IDF"] = y_pred


In [6]:
# Logistic Regression, using a pipline:
# Cuont Vector => TF_IDF Vector => Classifier

X = [" ".join(x) for x in combined["Lemmatized"]]
y = combined["Indicator"]

# Assign training and testing data
X_train = X[: len_train]
X_test = X[len_train :]
y_train = y.loc[: len_train - 1]
y_test = y.loc[len_train :]

logreg = Pipeline([('vect', CountVectorizer()),
                ('tfidf', TfidfTransformer()),
                ('clf', LogisticRegression(solver='lbfgs', max_iter=400)),
               ])

logreg.fit(X_train, y_train)
y_pred = logreg.predict(X_test)

#report_names = ["Reportable", "Non-reportable"]
#print("Accuracy:", accuracy_score(y_pred, y_test))
#print("\n------------------------------------------")
#print(  "Logistic Regression Classification Report:")
#print(  "------------------------------------------")
#print(classification_report(y_test, y_pred, target_names=report_names))

#ConfusionMatrix(y_test, y_pred)

modelResults["Logistic Regression - TF-IDF"] = y_pred


In [7]:
# Save model results

modelResults.to_csv("model_result.csv")

In [32]:
# For Naive Bayes - Count Vector model
y_true = modelResults["Actual Indicator"]
y_pred = modelResults["Naive Bayes - Count Vector"]

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print(f"Naive Bayes - Count Vector Model:")
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1 Score: {f1}")

KeyError: 'Actual Indicator'